In [1]:
import torch
import torch.nn as nn

# =====================================================================
# STEP 1: RESOLVE THE RUNTIME HARDWARE ENVIRONMENT DYNAMICALLY
# =====================================================================
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # Apple Silicon hardware verification hook
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("--- Hardware Deployment Initialization ---")
print(f"Active Accelerated Compute Environment Resolved Target: {device.type.upper()}\n")

# =====================================================================
# STEP 2: INSTANTIATE AND ROUTE NETWORK PARAMETERS
# =====================================================================
class DeploymentClassifier(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(in_features, 32),
            nn.ReLU(),
            nn.Linear(32, out_features)
        )
        
    def forward(self, x):
        return self.network(x)

model = DeploymentClassifier(in_features=10, out_features=2)

# Structural mutation: push model parameters to target chip
model.to(device)
print("--- Model Allocation Audit ---")
# Check the location of the first layer weight matrix to confirm successful routing
print(f"Model weight tensor memory address residency: {next(model.parameters()).device}\n")

# =====================================================================
# STEP 3: PROCESSSING DATA STREAM BATCHES SAFE FROM CRASHES
# =====================================================================
# Tensors are initialized on host CPU memory by default
cpu_batch_X = torch.randn(5, 10)
cpu_batch_y = torch.randint(0, 2, (5,))

print("--- Data Streaming Source Status ---")
print(f"Input features device: {cpu_batch_X.device} | Target labels device: {cpu_batch_y.device}")

# Routing tensors safely to match the device processing layer of the model
# Input tensors MUST match the model's device, or a RuntimeError will fire
device_batch_X = cpu_batch_X.to(device)
device_batch_y = cpu_batch_y.to(device)

print("\n--- Accelerated Data Transit Verification ---")
print(f"Routed features device: {device_batch_X.device} | Routed labels device: {device_batch_y.device}\n")

# Execute Forward Pass securely on the target hardware
model.eval()
with torch.no_grad():
    output_logits = model(device_batch_X)
    print("--- Execution Model Inference Complete ---")
    print(f"Output Matrix shape: {output_logits.shape} | Resident Hardware: {output_logits.device}")

# =====================================================================
# STEP 4: ROUTING METRICS BACK TO SYSTEM MEMORY FOR AUDITING
# =====================================================================
# If you attempt to call .numpy() on an accelerated tensor (CUDA/MPS), PyTorch 
# will throw an exception. You must pull the tensor back to CPU space first.
numpy_predictions = output_logits.cpu().numpy()
print("\n--- Downstream Output Collection ---")
print(f"Extracted Production Metrics Type: {type(numpy_predictions)}")

--- Hardware Deployment Initialization ---
Active Accelerated Compute Environment Resolved Target: CPU

--- Model Allocation Audit ---
Model weight tensor memory address residency: cpu

--- Data Streaming Source Status ---
Input features device: cpu | Target labels device: cpu

--- Accelerated Data Transit Verification ---
Routed features device: cpu | Routed labels device: cpu

--- Execution Model Inference Complete ---
Output Matrix shape: torch.Size([5, 2]) | Resident Hardware: cpu

--- Downstream Output Collection ---
Extracted Production Metrics Type: <class 'numpy.ndarray'>
